# 03 assignment

완전한 에이전트 루프를 갖춘 Movie Agent를 구축하세요!

에이전트가 수행해야 할 작업:
- 영화에 대한 사용자 질문을 받습니다.
- 어떤 도구를 호출할지 결정합니다 (필요한 경우).
- 실제로 API를 호출하여 LLM에게 실제 결과를 반환합니다.
- LLM이 최종 답변을 줄 때까지 루프를 계속합니다.

< 요구사항 > 
1. 실제 API를 호출하는 최소 3개의 도구를 갖춘 에이전트를 만드세요:
    - get_popular_movies() - /movies 호출
    - get_movie_details(id) - /movies/:id 호출
    - get_similar_movies(id) - /movies/:id/similar 호출

2. 에이전트가 갖춰야 할 조건:
    - (수동 프롬프팅이 아닌) OpenAI tools 파라미터를 사용하세요.
    - 응답에서 tool_calls를 처리하세요.
    - 실제 API를 호출하고 결과를 다시 전달해야 합니다.
    - 메모리를 활용한 멀티턴 대화를 지원해야 합니다.

3. 예시 상호작용

    User: 지금 인기 있는 영화 알려줘

    Agent: [get_popular_movies() 호출]

    Agent: 현재 인기 영화 목록입니다: 1. 듄: 파트 2 (ID: 693134)...

    
    User: 듄에 대해 더 알려줘

    Agent: [get_movie_details(693134) 호출]

    Agent: 듄: 파트 2는 드니 빌뇌브 감독의 2024년 SF 대작으로...

    User: 비슷한 영화 추천해 줄래?

    Agent: [get_similar_movies(693134) 호출]
    
    Agent: 듄을 좋아하셨다면 이런 영화를 추천드립니다: 블레이드 러너 2049, 어라이벌...

In [1]:
import requests, json

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    return requests.get(f"{BASE_URL}//movies/{id}").json()

def get_similar_movies(id):
    return requests.get(f"{BASE_URL}//movies/{id}/similar").json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록 가져옵니다",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "id로 영화 상세 정보 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of a movie"
                    }
                }
            },
            "required": ["id"]
        }
    },   
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "id와 유사한 영화를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of a movie"
                    }
                }
            },
            "required": ["id"]
        }
    },       
]



In [2]:
import openai
from openai.types.chat import ChatCompletionMessage


client = openai.Client()

messages = []

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant",
                "content":message.content or "",
                "tool_calls" : [
                    {
                        "id":tool_call.id,
                        "type":"function",
                        "function":{
                            "name":tool_call.function.name,
                            "arguments":tool_call.function.arguments,
                        }
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")
        
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
                print(f"Error: Invalid JSON arguments for function {function_name}, {arguments}")
                continue

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with {arguments} and get {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()

    else: 
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    # print(messages)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    # print(response.choices[0].message)
    process_ai_response(response.choices[0].message)

In [3]:
message = "지금 인기 있는 영화 알려줘"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: 지금 인기 있는 영화 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with {} and get [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg', 'genre_ids': [27, 53], 'id': 1339713, 'title': 'Obsession', 'original_language': 'en', 'original_title': 'Obsession', 'overview': 'After breaking the mysterious "One Wish Willow" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.', 'popularity': 817.0684, 'poster_path': 'https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg', 'release_date': '2026-05-13', 'softcore': False, 'video': False, 'vote_average': 7.9, 'vote_count': 707}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/oPsRr7AfNLw6XaPuMpvkWK0bIUA.jpg', 'genre_ids': [28, 18], 'id': 1057265, 'title': 'Peddi', 'original_language': 'te', 'original_title': 'పెద్ది', 'overview': 'In 1980s

In [4]:
message = "Obsession 대해 더 알려줘"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: Obsession 대해 더 알려줘
Calling function: get_movie_details with {"id":"1339713"}
Ran get_movie_details with {'id': '1339713'} and get {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg', 'belongs_to_collection': None, 'budget': 750000, 'genres': [{'id': 27, 'name': 'Horror'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'http://www.focusfeatures.com/obsession', 'id': 1339713, 'imdb_id': 'tt37287335', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Obsession', 'overview': 'After breaking the mysterious "One Wish Willow" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.', 'popularity': 817.0684, 'poster_path': 'https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg', 'production_companies': [{'id': 155758, 'logo_path': 'https://image.tmdb.org/t/p/w300/iZi0zd9ijrcWR9zlEUUthG8d600.png', 'name': 

In [5]:
message = "비슷한 영화 추천해 줄래?"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: 비슷한 영화 추천해 줄래?
Calling function: get_similar_movies with {"id":"1339713"}
Ran get_similar_movies with {'id': '1339713'} and get [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/t0krAUn3odB8H2KC60dJzKIWPyz.jpg', 'genre_ids': [27], 'id': 1355983, 'title': 'Scarecrow', 'original_language': 'tl', 'original_title': 'Espantaho', 'overview': "As a family clashes over their late father's land, an eerie painting appears and releases something sinister during the nine-day funeral ritual.", 'popularity': 0.7927, 'poster_path': 'https://image.tmdb.org/t/p/w780/itRDwuQxbtmgaxyrHI4rGiIrxxX.jpg', 'release_date': '2025-02-07', 'softcore': False, 'video': False, 'vote_average': 6.3, 'vote_count': 10}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/x7l1X3b7Dak6EMdi2LGAYkBjdmf.jpg', 'genre_ids': [53], 'id': 1355943, 'title': 'Above the Knee', 'original_language': 'no', 'original_title': 'Over kneet', 'overview': 'Amir has a secret. He’s tormented by visions 